# Snapget — tự động cào ảnh + tự động gán nhãn cho tập VN-train (Google Colab)

Chạy `ml/scripts/17_crawl_autolabel.py` theo **`ml/DATASET_SPEC.md`**: tìm ảnh theo từ khoá bối cảnh Việt Nam (`ml/crawl_queries.yaml`) trên **Openverse + Wikimedia Commons** (ảnh Creative Commons, có license) và **DuckDuckGo** (tìm tiếng Việt) → lọc chất lượng (cạnh dài ≥ 1024, bỏ nền trắng studio, bỏ trang stock/TMĐT hay có watermark) → khử trùng → **gán box bằng 2 model** (YOLO11x COCO + YOLO-World với prompt "ghế nhựa", "ly trà sữa", "xe ga"… và hard negative "đồng hồ đeo tay", "nón lá"…) → ảnh chắc chắn vào `train/`, ảnh nghi vấn vào `review/` để người sửa trên Roboflow.

**Runtime → Change runtime type → T4 GPU.** Kết quả lưu thẳng lên Drive (`WORK/data/vn_web/`) → Colab ngắt thì **chạy lại ô 3, script tự chạy tiếp** (bỏ URL đã xử lý, đếm ảnh đã có).

> ⚠️ Ảnh Internet **chỉ vào VN-train**, không bao giờ vào test. Ảnh `accept` vào thẳng train không qua người duyệt từng ảnh → **xem mẫu ~10%** ở ô 4 (DATASET_SPEC mục G). Không cào trực tiếp Google Images (Google chặn IP Colab + cấm trong điều khoản) — DuckDuckGo cho kết quả tương đương.

In [ ]:
# 0. Drive + code + thư viện
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/snapget-ml'
OUT = f'{WORK}/data/vn_web'          # thư mục VN của ảnh cào (chỉ có train/) — dùng ở notebook train: --vn {OUT}
import os; os.makedirs(OUT, exist_ok=True)
%cd /content
!rm -rf datn && git clone --depth 1 https://github.com/HuyenVt001/datn.git datn
%cd /content/datn/ml
!pip install -q ultralytics ddgs requests pyyaml
!nvidia-smi --query-gpu=name,memory.total --format=csv

1. Ảnh TEST thật chống rò rỉ

In [ ]:
import os
import glob

TEST_ARGS = ''
r = '/content/drive/MyDrive/snapget-ml/data/vn.rar'

if os.path.exists(r):
    !mkdir -p /content/ext_vn
    !unrar x -o+ -inul "{r}" /content/ext_vn/
    tests = [d for d in glob.glob('/content/ext_vn/**/test/images', recursive=True)]
    TEST_ARGS = ' '.join(f'--test-dir {d}' for d in tests)

print('test dirs:', TEST_ARGS or '(chưa có — bỏ qua)')

## 2. (Tuỳ chọn) Token Openverse — giới hạn tốc độ cao hơn
Ẩn danh vẫn chạy được nhưng bị giới hạn số request. Đăng ký miễn phí: `POST https://api.openverse.org/v1/auth_tokens/register/` (xem docs.openverse.org) → xác nhận email → được `client_id`/`client_secret`. **Không commit, không dán vào chat** — đặt vào Colab *Secrets* (biểu tượng 🔑 bên trái) tên `OPENVERSE_CLIENT_ID`, `OPENVERSE_CLIENT_SECRET`.

In [ ]:
try:
    from google.colab import userdata
    for k in ('OPENVERSE_CLIENT_ID', 'OPENVERSE_CLIENT_SECRET'):
        v = userdata.get(k)
        if v:
            os.environ[k] = v
    print('Openverse token:', 'có' if os.environ.get('OPENVERSE_CLIENT_ID') else 'không (ẩn danh)')
except Exception:
    print('Openverse: ẩn danh')

## 3. Cào + gán nhãn — chạy lại ô này để chạy TIẾP sau khi Colab ngắt
- `--per-class 100`: số ảnh **accept** cần có mỗi lớp (đặc tả: VN-train ≥ 150/lớp, trong đó ≥ 50 tự chụp ⇒ ảnh cào ~100).
- `--negatives 150 --hard-negatives 150` (đặc tả: 300 negative, ≥ 150 hard).
- Thử nhanh 1 lớp trước: `--classes chair --per-class 10`.
- Openverse hay báo `504` (server của họ quá tải): script tự thử lại, lỗi 3 lần liên tiếp thì tạm tắt 15 phút. Muốn bỏ hẳn: `--sources ddg,wikimedia`.
- Thiếu ảnh 1 lớp → thêm từ khoá trong `ml/crawl_queries.yaml` (fork/sửa repo) hoặc tăng `--per-query`.

In [ ]:
!python scripts/17_crawl_autolabel.py --out {OUT} --per-class 100 --negatives 150 --hard-negatives 150 \
    --sources ddg,wikimedia,openverse --per-query 40 --device 0 {TEST_ARGS}

## 4. Xem mẫu ~10% ảnh accept 
Box đỏ = nhãn tự động. Thấy sai nhiều ở lớp nào → báo Claude / tăng `--accept` (chặt hơn) hoặc đưa lớp đó qua review.

In [ ]:
import random, sys
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display
sys.path.insert(0, '/content/datn/ml')
from snapget12 import CLASSES
from snapget12.yolo import read_yolo_labels, find_names
names = find_names(OUT) or CLASSES
imgs = sorted(Path(f'{OUT}/train/images').glob('gg_*.jpg'))
sample = random.sample(imgs, min(len(imgs), max(12, len(imgs) // 10)))
for i in range(0, len(sample), 4):
    tiles = []
    for p in sample[i:i + 4]:
        im = Image.open(p).convert('RGB'); im.thumbnail((400, 400)); d = ImageDraw.Draw(im); W, H = im.size
        for c, x, y, w, h in read_yolo_labels(Path(f'{OUT}/train/labels') / (p.stem + '.txt')):
            d.rectangle([(x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H], outline=(255, 0, 0), width=3)
            d.text(((x - w / 2) * W + 3, (y - h / 2) * H + 3), names[c], fill=(255, 255, 0))
        d.text((4, H - 14), p.name, fill=(0, 255, 0)); tiles.append(im)
    grid = Image.new('RGB', (1600, 400), (30, 30, 30))
    for k, t in enumerate(tiles):
        grid.paste(t, (k * 400, 0))
    display(grid)

## 5. Ảnh REVIEW → sửa trên Roboflow
1. Ô dưới zip `review/` (images + labels + classes.txt) lên Drive.
2. Roboflow → *Create Project* (Object Detection) → *Upload* zip → box nháp hiện sẵn. Xem lý do từng ảnh trong `review/review_queue.csv` (vd "box điểm thấp", "umbrella trùng 'conical hat'", "laptop chưa có box keyboard"). Sửa theo DATASET_SPEC mục B, D; **xoá** ảnh không dùng được.
3. *Generate version*: **không** Preprocessing/Augmentation, 100% train → *Export* YOLOv8 → tải zip về, kéo lên Drive `WORK/data/review_fixed.zip`.

In [ ]:
!cd {OUT} && zip -q -r {WORK}/data/vn_web_review.zip review -x 'review/metadata_pending.csv'
!ls -la {WORK}/data/vn_web_review.zip && head -5 {OUT}/review/review_queue.csv

## 6. Đưa ảnh đã sửa vào train (`--promote`)
Dùng ảnh gốc độ phân giải cao trong `review/` + nhãn từ bản export (tên file `.rf.xxx` của Roboflow được xử lý tự động).

In [ ]:
!rm -rf /content/review_fixed && unzip -q -o {WORK}/data/review_fixed.zip -d /content/review_fixed
!python scripts/17_crawl_autolabel.py --out {OUT} --promote /content/review_fixed

## 7. Báo cáo
`crawl_report.json` (số ảnh, lý do lọc/loại, độ đa dạng tối/xa/bối cảnh từng lớp), `data_sources.md` + `web_attributions.csv` (trích nguồn cho báo cáo). Dùng trong notebook train: ô 4 thêm `--vn {OUT}` (ảnh cào) bên cạnh `--vn` ảnh tự chụp.

In [ ]:
import json
r = json.load(open(f'{OUT}/crawl_report.json'))
print('accept:', r['accepted']); print('review:', r['review'])
for g, v in r['runs'].items():
    print(g, '| loai:', dict(sorted(v['reject'].items(), key=lambda x: -x[1])[:3]), '| loc:', dict(sorted(v['filtered'].items(), key=lambda x: -x[1])[:4]))
print(open(f'{OUT}/data_sources.md').read())